kernel : Python (Pixi)

In [1]:
%load_ext autoreload
%autoreload 2
import anndata as ad
import gc
import os
import pandas as pd
import scanpy as sc
from anndata import AnnData
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir
from pipeline.utils.get_ensembl_to_symbol import get_ensg_to_symbol

raw_data_dir = find_env_dir("RAW_DATA")
pre_h5ad_dir = find_env_dir("PRE_H5AD")

def load_10x_data(mtx_file: str, barcodes_file: str, features_file: str) -> AnnData:
    adata: AnnData = sc.read_mtx(mtx_file).T

    with open(barcodes_file, "r") as f:
        barcodes = [line.strip().split("\t")[0] for line in f]
    with open(features_file, "r") as f:
        genes = [line.strip().split("\t")[0] for line in f]
    adata.obs_names = barcodes
    adata.var_names = genes

    if not isinstance(adata.X, csr_matrix):
        adata.X = csr_matrix(adata.X)
    return adata

In [2]:
SERIES = "kong"
SAVE_FILE = SERIES + "_raw.h5ad"
EXPRESSION = "expression"
data_dir = os.path.join(raw_data_dir, SERIES, "SCP1884")

colon_stromal_dir = os.path.join(data_dir, EXPRESSION, "62a0c9f523a28233e1c9b06b")
COLON_STROMAL_MTX = "CO_STR.scp.raw.mtx"
COLON_STROMAL_BARCODES = "CO_STR.scp.barcodes.tsv"
COLON_STROMAL_GENES = "CO_STR.scp.features.tsv"

colon_stromal_adata = load_10x_data(
    os.path.join(colon_stromal_dir, COLON_STROMAL_MTX),
    os.path.join(colon_stromal_dir, COLON_STROMAL_BARCODES),
    os.path.join(colon_stromal_dir, COLON_STROMAL_GENES),
)

colon_immune_dir = os.path.join(data_dir, EXPRESSION, "62a7a911a54b79c09baa336a")
COLON_IMMUNE_MTX = "CO_IMM.scp.raw.mtx"
COLON_IMMUNE_BARCODES = "CO_IMM.scp.barcodes.tsv"
COLON_IMMUNE_GENES = "CO_IMM.scp.features.tsv"

colon_immune_adata = load_10x_data(
    os.path.join(colon_immune_dir, COLON_IMMUNE_MTX),
    os.path.join(colon_immune_dir, COLON_IMMUNE_BARCODES),
    os.path.join(colon_immune_dir, COLON_IMMUNE_GENES),
)

colon_epithelial_dir = os.path.join(data_dir, EXPRESSION, "62a79393d8bced7ddefbf0d1")
COLON_EPITHELIAL_MTX = "CO_EPI.scp.raw.mtx"
COLON_EPITHELIAL_BARCODES = "CO_EPI.scp.barcodes.tsv"
COLON_EPITHELIAL_GENES = "CO_EPI.scp.features.tsv"

colon_epithelial_adata = load_10x_data(
    os.path.join(colon_epithelial_dir, COLON_EPITHELIAL_MTX),
    os.path.join(colon_epithelial_dir, COLON_EPITHELIAL_BARCODES),
    os.path.join(colon_epithelial_dir, COLON_EPITHELIAL_GENES),
)

colon_adata = ad.concat(
    {
        "stromal": colon_stromal_adata,
        "immune": colon_immune_adata,
        "epithelial": colon_epithelial_adata
    }, 
    label="celltype_coarse",
    join="outer",
    fill_value=0
)

ileum_stromal_dir = os.path.join(data_dir, EXPRESSION, "62a7b26f0e85c7e6d0bb03f0")
ILEUM_STROMAL_MTX = "TI_STR.scp.raw.mtx"
ILEUM_STROMAL_BARTIDES = "TI_STR.scp.barcodes.tsv"
ILEUM_STROMAL_GENES = "TI_STR.scp.features.tsv"

ileum_stromal_adata = load_10x_data(
    os.path.join(ileum_stromal_dir, ILEUM_STROMAL_MTX),
    os.path.join(ileum_stromal_dir, ILEUM_STROMAL_BARTIDES),
    os.path.join(ileum_stromal_dir, ILEUM_STROMAL_GENES),
)

ileum_immune_dir = os.path.join(data_dir, EXPRESSION, "62a7c0fd3c8dbbf858d3ac47")
ILEUM_IMMUNE_MTX = "TI_IMM.scp.raw.mtx"
ILEUM_IMMUNE_BARTIDES = "TI_IMM.scp.barcodes.tsv"
ILEUM_IMMUNE_GENES = "TI_IMM.scp.features.tsv"

ileum_immune_adata = load_10x_data(
    os.path.join(ileum_immune_dir, ILEUM_IMMUNE_MTX),
    os.path.join(ileum_immune_dir, ILEUM_IMMUNE_BARTIDES),
    os.path.join(ileum_immune_dir, ILEUM_IMMUNE_GENES),
)

ileum_epithelial_dir = os.path.join(data_dir, EXPRESSION, "62a7b6bf52be218275f15f43")
ILEUM_EPITHELIAL_MTX = "TI_EPI.scp.raw.mtx"
ILEUM_EPITHELIAL_BARTIDES = "TI_EPI.scp.barcodes.tsv"
ILEUM_EPITHELIAL_GENES = "TI_EPI.scp.features.tsv"

ileum_epithelial_adata = load_10x_data(
    os.path.join(ileum_epithelial_dir, ILEUM_EPITHELIAL_MTX),
    os.path.join(ileum_epithelial_dir, ILEUM_EPITHELIAL_BARTIDES),
    os.path.join(ileum_epithelial_dir, ILEUM_EPITHELIAL_GENES),
)

ileum_adata = ad.concat(
    {
        "stromal": ileum_stromal_adata,
        "immune": ileum_immune_adata,
        "epithelial": ileum_epithelial_adata
    }, 
    label="celltype_coarse",
    join="outer",
    fill_value=0
)

adata = ad.concat(
    [colon_adata, ileum_adata], 
    join="outer",
    fill_value=0
)

META_DATA = "scp_metadata_combined.v2.txt"
meta = pd.read_csv(os.path.join(data_dir, "metadata", META_DATA), sep="\t", skiprows=[1])
meta.set_index("NAME", inplace=True)

assert isinstance(adata.obs, pd.DataFrame)
adata.obs = adata.obs.join(meta, how="left")

adata.obs.head() #type: ignore

/tmp/ipykernel_2097177/3893468271.py:101: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(os.path.join(data_dir, "metadata", META_DATA), sep="\t", skiprows=[1])


,celltype_coarse,biosample_id,n_genes,n_counts,Chem,Site,Type,donor_id,Layer,Celltype,sex,species,species__ontology_label,library_preparation_protocol,library_preparation_protocol__ontology_label,organ,organ__ontology_label,disease,disease__ontology_label
N105446_L-GTGTGGCTCCGTCAAA,stromal,N105446_L,5135,19014,v3,CO,NonI,105446,L,Fibroblasts ADAMDEC1,unknown,NCBITaxon_9606,Homo sapiens,EFO_0009922,10x 3' v3,UBERON_0001155,colon,MONDO_0005011,Crohn's disease
N105446_L-CAATACGAGTCCCTAA,stromal,N105446_L,5119,18425,v3,CO,NonI,105446,L,Endothelial cells CD36,unknown,NCBITaxon_9606,Homo sapiens,EFO_0009922,10x 3' v3,UBERON_0001155,colon,MONDO_0005011,Crohn's disease
N105446_L-CCCTGATAGTGTTCCA,stromal,N105446_L,5024,18305,v3,CO,NonI,105446,L,Fibroblasts ADAMDEC1,unknown,NCBITaxon_9606,Homo sapiens,EFO_0009922,10x 3' v3,UBERON_0001155,colon,MONDO_0005011,Crohn's disease
N105446_L-CATTGTTAGAGCCCAA,stromal,N105446_L,4817,17791,v3,CO,NonI,105446,L,Fibroblasts ADAMDEC1,unknown,NCBITaxon_9606,Homo sapiens,EFO_0009922,10x 3' v3,UBERON_0001155,colon,MONDO_0005011,Crohn's disease
N105446_L-TCCATGCGTTCGTTCC,stromal,N105446_L,4690,17548,v3,CO,NonI,105446,L,Fibroblasts KCNN3 LY6H,unknown,NCBITaxon_9606,Homo sapiens,EFO_0009922,10x 3' v3,UBERON_0001155,colon,MONDO_0005011,Crohn's disease


In [3]:
assert isinstance(adata.obs, pd.DataFrame)
adata.obs["celltype_coarse"] = adata.obs["celltype_coarse"]
adata.obs["sample"] = adata.obs["biosample_id"]
adata.obs["prep"] = adata.obs["Chem"]
adata.obs["site"] = adata.obs["Site"]
adata.obs["condition"] = adata.obs["Type"]
adata.obs["donor"] = adata.obs["donor_id"].astype(str)
adata.obs["layer"] = adata.obs["Layer"]
adata.obs["celltype"] = adata.obs["Celltype"]
adata.obs["organ"] = adata.obs["organ__ontology_label"]
adata.obs["disease"] = adata.obs["disease__ontology_label"]
adata.obs["series"] = SERIES

keep = ["celltype_coarse", "sample", "prep", "site", "condition", "donor", "layer", "celltype", "organ", "disease", "series"]
adata.obs.drop(columns = [c for c in adata.obs.columns if c not in keep], inplace=True) #type: ignore

In [4]:
ensg_to_symbol = get_ensg_to_symbol()

vn = pd.Series(adata.var_names, index=adata.var_names).astype("string")
ensg = vn.str.split(".", n=1).str[0]

bad = ~ensg.str.match(r"^ENSG\d+$", na=False)
if bad.any():
    print("[WARN] Non-ENSG-like var_names examples:")
    print(vn[bad][:20].tolist())

mapped = ensg.map(ensg_to_symbol).replace("", pd.NA).fillna(vn)

adata.var["orig_var_name"] = adata.var_names.astype(str).tolist() 
adata.var_names = mapped.values.astype(str).tolist() 
adata.var_names_make_unique()

In [6]:
adata.obs.index = adata.obs.index.astype(str)
adata.var.index = adata.var.index.astype(str)
adata.write_h5ad(os.path.join(pre_h5ad_dir, SAVE_FILE))
del adata
gc.collect()

36